In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost tqdm -q

clear_output()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
from catboost import CatBoostClassifier
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score


warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
# Task 1: Write your code here:
file_path = os.path.join(path, 'Q3_data.csv')

df = pd.read_csv(file_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
display(df.info())
df.shape

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")

print('target column: ', df['Target'].isnull().sum())
''' Not missing values in the target column '''
display(missing_data)
above50 = list(missing_data[missing_data['Missing_Percentage'] >= 50].values[:, 0])
lessthan50 = list(missing_data[missing_data['Missing_Percentage'] < 50].values[:, 0])

df.drop(above50,axis=1, inplace=True)

for col in lessthan50:
  df[col].fillna(df[col].mean(), inplace=True)

print('remaining nan: ', int(df.isnull().sum().sum()))


In [ ]:
# Task 2: Write your code here:
dups = int(df.duplicated().sum())
print('Number of duplicated rows = ', dups)

''' No duplicated rows '''

In [ ]:
# Task 3: Write your code here:
target_column = 'Target'
categorical_columns = list(df.select_dtypes('object').columns)
print('Categorical columns: ', categorical_columns)

''' No categorical columns no need to label encoding or one hot encoding'''

numerical_columns = list(df.select_dtypes(include = ['int64', 'float64']).columns.drop(target_column)) # Will be used later for scaling..
print('Numerical columns: ', numerical_columns)

''' as a sanity check I will equate the number of numerical columns to the num of features in the dataset '''
print(len(numerical_columns) == (df.shape[1] - 1)) # df.shape[1] - 1 disgarding the target

In [ ]:
# Task 4: Write your code here:
scaler = StandardScaler()

df[numerical_columns] = scaler.fit_transform(df[numerical_columns])
df

In [ ]:
# Task 5: Write your code here:
plt.figure()
sns.countplot(x=df[target_column])
plt.show()
print(df[target_column].value_counts(normalize=True))

''' Target is clearly imbalance, thus, i will use stratified kfold and f1 score '''

In [ ]:
# Task 1: Write your code here:
X = df.drop(target_column, axis=1)
y = df[target_column]

In [ ]:
# Task 2,3,4,5: Write your code here:
n_splits = 5

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

model = CatBoostClassifier(verbose=0, n_estimators=200, max_depth=4)

lr_f1 = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  print(f"Training CatBoostClassifier...")

  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)

  f1 = f1_score(y_test, y_pred, average='macro')

  lr_f1.append(f1)

print('\n Average score = ', np.mean(lr_f1))

In [ ]:
# Task 1: Write your code here:
CatBoost_model = model
CatBoost_importance = list(zip(X.columns, CatBoost_model.feature_importances_))
sorted_CatBoost_importance = sorted(CatBoost_importance, key=lambda x: abs(x[1]), reverse=True)

# Extract features and their coefficients
features, coefficients = zip(*sorted_CatBoost_importance)

# Plot feature importances
plt.figure(figsize=(10, 6))
plt.barh(features, coefficients, color='darkred')
plt.xlabel('Coefficient Value')
plt.ylabel('Features')
plt.title('CatBoost Classifier Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
print(sorted_CatBoost_importance[0])

In [ ]:
# Task Bonus: Write your code here: